In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
sixmetrics_from_SUB_variables_composite5.py
-------------------------------------------
for_checking_20251127/sub 下の STEP3.2（signlessCorr_MDS）結果を使い、

  ・ cluster_labels_matrix_variables_{OH,FP}_*.csv   （variables / OH, FP）
  ・ for_MDS_STEP2/preprocessed_features_{OH,FP}.csv （列=変数名）

から、

  1) OH→FP / FP→OH の 6 指標
      - mean_purity
      - pair_major_same_rate
      - pair_mean_cosine_similarity
      - cluster_median_cohesive_ratio
      - mean_entropy
      - pair_mean_JS_divergence
      + 正方向にそろえた entropy_coh, js_coh
      + composite5 = (purity, major_same, cosine, cohesive, entropy_coh) の平均

  2) NbClust 6 指標 × (top3 / cumeig) について
      - 方向別（OH→FP / FP→OH）の 12 本棒グラフ
      - 両方向を並べた 24 本棒グラフ

を PNG/PDF と CSV に出力する。

想定ディレクトリ:
  BASE (sub) = .../for_checking_20251127/sub
    └─ 03_clustering_STEP3.2_signlessCorr/
         └─ run_YYYYMMDD_HHMMSS/
             └─ variables/
                 ├─ OH/cluster_labels_matrix_variables_OH_*.csv
                 └─ FP/cluster_labels_matrix_variables_FP_*.csv

  FEAT_ROOT = .../for_checking_20251127/data/for_MDS_STEP2
    ├─ preprocessed_features_OH.csv
    └─ preprocessed_features_FP.csv

出力:
  <BASE>/plots_sixmetrics_sub/
    ├─ Table_sixmetrics_bidirectional_composite5_sub.csv
    ├─ Fig_OHtoFP_<metric>_top3_cumeig.png/pdf
    ├─ Fig_FPtoOH_<metric>_top3_cumeig.png/pdf
    ├─ Fig_compareDirections_<metric>_top3_cumeig.png/pdf

※ 注意:
  - cluster_labels_matrix は linear/nonlinear の両方を含むが、
    現段階では「linear_*」列のみを使用する（cmdscale ベース）。
"""

from pathlib import Path
import argparse, re, itertools

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =========================
# 既定パス
# =========================
DEFAULT_BASE = Path(
    "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/"
    "cal_cluster_No/20251026_under_25clusters_program_and_result/"
    "for_checking_20251127/sub"
)

DEFAULT_FEATROOT = Path(
    "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/"
    "cal_cluster_No/20251026_under_25clusters_program_and_result/"
    "for_checking_20251127/data/for_MDS_STEP2"
)

# =========================
# 解析設定
# =========================
IDX_ORDER = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
MODES     = ["top3", "cumeig"]
TARGETS   = {"PCEmax", "Jsc", "Voc", "FF"}  # 目的変数（列から除外）

# 色設定（direction × mode）
COLOR_OH_TOP3    = "#1f77b4"  # OH→FP, top3 (deep blue)
COLOR_OH_CUMEIG  = "#aec7e8"  # OH→FP, cumeig (light blue)
COLOR_FP_TOP3    = "#d62728"  # FP→OH, top3 (deep red)
COLOR_FP_CUMEIG  = "#ff9896"  # FP→OH, cumeig (light red / salmon)


def debug(msg: str):
    print(f"[DEBUG] {msg}")


# =========================
# cluster_labels_matrix の探索 & 読み込み
# =========================
def find_cluster_labels_matrix(base: Path, dataset: str) -> Path | None:
    """
    sub/03_clustering_STEP3.2_signlessCorr 以下を再帰探索して、
    cluster_labels_matrix_variables_{dataset}_*.csv を探す。

    複数見つかった場合は、更新日時が最新のものを採用。
    """
    pattern = f"cluster_labels_matrix_variables_{dataset}_*.csv"
    cand = list(base.rglob(pattern))
    if not cand:
        print(f"[WARN] cluster_labels_matrix not found for dataset={dataset} under {base}")
        return None
    # 更新時刻でソート（最新を採用）
    cand.sort(key=lambda p: p.stat().st_mtime)
    best = cand[-1]
    print(f"[INFO] Using cluster_labels_matrix for {dataset}: {best}")
    return best


def melt_cluster_labels(df: pd.DataFrame,
                        mds_method: str = "linear") -> pd.DataFrame:
    """
    cluster_labels_matrix_variables_*.csv を縦持ち化する。
    """
    # 念のためもう一度クリーニング
    df.columns = [str(c).strip().lstrip("\ufeff") for c in df.columns]

    # ★ 'id' が無い場合は「1 列目を id とみなしてリネーム」
    if "id" not in df.columns:
        first_col = df.columns[0]
        print(f"[WARN] 'id' column not found. Use '{first_col}' as id.")
        df = df.rename(columns={first_col: "id"})

    # ここから先は元のコードと同じ
    use_cols = ["id"] + [c for c in df.columns if c.startswith(f"{mds_method}_")]
    df = df.loc[:, use_cols].copy()

    rows = []
    for col in df.columns:
        if col == "id":
            continue
        parts = col.split("_")
        if len(parts) != 3:
            debug(f"  skip column (unexpected name): {col}")
            continue
        method, mode, index = parts
        if mode not in MODES or index not in IDX_ORDER:
            debug(f"  skip column (unknown mode/index): {col}")
            continue

        for i in range(len(df)):
            var_id = str(df.iloc[i]["id"])
            val = df.iloc[i][col]
            if pd.isna(val):
                continue
            try:
                cl = int(val)
            except Exception:
                continue
            rows.append({
                "id": var_id,
                "cluster": cl,
                "mode": mode,
                "index": index
            })

    if not rows:
        debug("melt_cluster_labels: no valid rows created.")
        return pd.DataFrame(columns=["id", "cluster", "mode", "index"])

    A = pd.DataFrame(rows)
    A["id"] = A["id"].astype(str)
    A["cluster"] = A["cluster"].astype(int)
    return A
def read_assignments_from_sub(base: Path,
                              dataset: str,
                              mds_method: str = "linear") -> pd.DataFrame:
    """
    sub 用：
      - cluster_labels_matrix_variables_{dataset}_*.csv を探して読み込み
      - melt して「id, cluster, mode, index」に変換
    """
    path = find_cluster_labels_matrix(base, dataset)
    if path is None:
        return pd.DataFrame(columns=["id", "cluster", "mode", "index"])

    df = pd.read_csv(path)
    A = melt_cluster_labels(df, mds_method=mds_method)
    print(f"[INFO] Assignments ({dataset}, method={mds_method}): {A.shape}")
    return A


# =========================
# 特徴量（変数ベクトル）読み込み
# =========================
def load_features_columns_as_variables(path: Path) -> pd.DataFrame:
    """
    preprocessed_features_{OH,FP}.csv を読み、列=変数名 を対象にするため転置。
    index（=変数名）を文字列統一、目的変数（PCEmax,Jsc,Voc,FF）は除外。
    NaN → 0、負値は 0 にクリップしてから行方向で確率化に使用。
    """
    if not path.exists():
        debug(f"Feature file missing: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, header=0, index_col=0)

    # 目的変数を列から除外
    keep = [c for c in df.columns if c not in TARGETS]
    df = df.loc[:, keep]

    # 変数方向（列）を対象にするので転置（index=変数名、列=サンプル）
    df = df.T

    # 数値変換
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # NaN→0, 負値→0（頻度のように扱う）
    df = df.fillna(0.0)
    df[df < 0.0] = 0.0
    df.index = df.index.astype(str)
    return df


def row_normalize_to_prob(X: pd.DataFrame) -> pd.DataFrame:
    """各行（=変数）の分布を確率化（sum=1）。全0は0のまま。"""
    S = X.sum(axis=1)
    S = S.replace(0, np.nan)
    Y = X.div(S, axis=0).fillna(0.0)
    return Y


# =========================
# 距離・情報量系ヘルパ
# =========================
def cosine(u: np.ndarray, v: np.ndarray) -> float:
    nu = np.linalg.norm(u)
    nv = np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return 0.0
    return float(np.dot(u, v) / (nu * nv))


def js_divergence(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen–Shannon divergence with log2."""
    m = 0.5 * (p + q)

    def kl(a, b):
        nz = (a > 0) & (b > 0)
        return np.sum(a[nz] * np.log2(a[nz] / b[nz]))

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def entropy_from_counts(counts: pd.Series) -> float:
    tot = counts.sum()
    if tot <= 0:
        return 0.0
    p = counts[counts > 0] / tot
    return float(-np.sum(p * np.log2(p)))


# =========================
# 6 指標（1方向）
# =========================
def six_metrics_direction(left_assign: pd.DataFrame,
                          right_assign: pd.DataFrame,
                          right_vec_prob: pd.DataFrame,
                          mode: str,
                          index: str,
                          thr=(0.6, 0.6, 0.5)):
    """
    left_assign のクラスタ（id=変数名）を right 側で眺めたときの整合度（6指標）

      - mean_purity:
          Lクラスタ内でのRラベル分布の最大比率の平均

      - pair_major_same_rate:
          Lクラスタ内ペアで Rラベル一致の割合

      - pair_mean_cosine_similarity:
          Lクラスタ内ペアの right_vec 確率分布のコサイン平均

      - cluster_median_cohesive_ratio:
          Lクラスタ内ペアで
            (R同一 & purity>=thr[0]) or (cos>=thr[1]) or (JS<=thr[2])
          を満たすペアの割合の中央値

      - mean_entropy:
          Rラベル分布のシャノンエントロピーの平均

      - pair_mean_JS_divergence:
          Lクラスタ内ペア分布の JS 平均
    """
    L = left_assign[(left_assign["mode"] == mode) & (left_assign["index"] == index)][["id", "cluster"]].rename(columns={"cluster": "L"})
    R = right_assign[(right_assign["mode"] == mode) & (right_assign["index"] == index)][["id", "cluster"]].rename(columns={"cluster": "R"})
    M = pd.merge(L, R, on="id", how="inner")
    if M.empty:
        return None

    pur_list, ent_list, cos_vals, js_vals, coh_ratio = [], [], [], [], []
    pair_same_n = 0
    pair_tot_n = 0

    for l, grp in M.groupby("L"):
        # Rラベル分布 → purity / entropy
        vc = grp["R"].value_counts()
        purity = vc.max() / vc.sum() if vc.sum() > 0 else np.nan
        ent = entropy_from_counts(vc)
        pur_list.append(purity)
        ent_list.append(ent)

        # L内ペアでコサイン・JS・R同一・cohesive
        ids = grp["id"].tolist()
        if len(ids) >= 2:
            sub = right_vec_prob.loc[right_vec_prob.index.intersection(ids)]
            mat = sub.to_numpy()
            same = 0
            total = 0
            coh = 0
            for i, j in itertools.combinations(range(mat.shape[0]), 2):
                total += 1
                same += int(grp["R"].iloc[i] == grp["R"].iloc[j])
                c = cosine(mat[i], mat[j])
                js = js_divergence(mat[i], mat[j])
                cos_vals.append(c)
                js_vals.append(js)
                cond1 = (grp["R"].iloc[i] == grp["R"].iloc[j]) and (purity >= thr[0])
                cond2 = (c >= thr[1])
                cond3 = (js <= thr[2])
                if cond1 or cond2 or cond3:
                    coh += 1
            pair_same_n += same
            pair_tot_n += total
            if total > 0:
                coh_ratio.append(coh / total)

    return {
        "mean_purity": np.nanmean(pur_list) if pur_list else np.nan,
        "pair_major_same_rate": (pair_same_n / pair_tot_n) if pair_tot_n > 0 else np.nan,
        "pair_mean_cosine_similarity": np.nanmean(cos_vals) if cos_vals else np.nan,
        "cluster_median_cohesive_ratio": np.nanmedian(coh_ratio) if coh_ratio else np.nan,
        "mean_entropy": np.nanmean(ent_list) if ent_list else np.nan,
        "pair_mean_JS_divergence": np.nanmean(js_vals) if js_vals else np.nan,
        "k_left": M["L"].nunique(),
        "k_right": M["R"].nunique(),
        "n": len(M),
    }


# =========================
# 図化: 方向別 12 本棒グラフ
# =========================
def save_bar_12(df_one_direction: pd.DataFrame,
                metric: str,
                title: str,
                direction: str,
                outbase: Path):
    """
    direction: "OH→FP" or "FP→OH"
    x 軸: NbClust 指標（6 個）
    棒: top3 / cumeig の 2 本
    x 軸下に left/right の k を表示（top3 の k を代表値として利用）
    """
    # 欠損列ガード
    for col in ["k_left", "k_right", metric]:
        if col not in df_one_direction.columns:
            df_one_direction[col] = np.nan

    if direction == "OH→FP":
        left_name, right_name = "OH", "FP"
    else:
        left_name, right_name = "FP", "OH"

    x = np.arange(len(IDX_ORDER), dtype=float)
    width = 0.35

    vals_top3, vals_cume = [], []
    k_left_top3, k_right_top3 = [], []

    for ix in IDX_ORDER:
        row_t = df_one_direction[(df_one_direction["index"] == ix) & (df_one_direction["mode"] == "top3")]
        row_c = df_one_direction[(df_one_direction["index"] == ix) & (df_one_direction["mode"] == "cumeig")]

        if row_t.empty:
            vals_top3.append(np.nan)
            k_left_top3.append(np.nan)
            k_right_top3.append(np.nan)
        else:
            vals_top3.append(float(row_t.iloc[0][metric]) if pd.notna(row_t.iloc[0][metric]) else np.nan)
            k_left_top3.append(row_t.iloc[0]["k_left"])
            k_right_top3.append(row_t.iloc[0]["k_right"])

        if row_c.empty:
            vals_cume.append(np.nan)
        else:
            vals_cume.append(float(row_c.iloc[0][metric]) if pd.notna(row_c.iloc[0][metric]) else np.nan)

    fig, ax = plt.subplots(figsize=(10.0, 5.0), dpi=300)

    if direction == "OH→FP":
        color_top = COLOR_OH_TOP3
        color_cum = COLOR_OH_CUMEIG
    else:
        color_top = COLOR_FP_TOP3
        color_cum = COLOR_FP_CUMEIG

    ax.bar(x - width / 2, vals_top3, width=width, color=color_top, label="top3")
    ax.bar(x + width / 2, vals_cume, width=width, color=color_cum, label="cumeig")

    ax.set_xticks(x)
    ax.set_xticklabels(IDX_ORDER, rotation=0, ha="center")
    ax.set_ylabel(metric)
    ax.set_title(title)

    # y 軸範囲（見やすさ重視）
    finite_vals = [v for v in (vals_top3 + vals_cume) if np.isfinite(v)]
    if finite_vals:
        vmax = max(finite_vals)
    else:
        vmax = 1.0

    if metric == "pair_mean_JS_divergence":
        ytop = float(min(max(vmax * 1.20, 0.2), 2.0))
        ax.set_ylim(0.0, ytop)
    elif metric == "mean_entropy":
        ytop = float(min(max(vmax * 1.20, 0.5), 6.0))
        ax.set_ylim(0.0, ytop)
    else:
        ytop = float(min(max(vmax * 1.10, 0.4), 1.05))
        ax.set_ylim(0.0, ytop)

    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5, axis="y")

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.0)

    # x 軸下に left/right k を表示（top3 の k_left/k_right を代表値として利用）
    for i, ix in enumerate(IDX_ORDER):
        kl = k_left_top3[i]
        kr = k_right_top3[i]
        if pd.isna(kl) or pd.isna(kr):
            continue

        if direction == "OH→FP":
            k_left, k_right = kl, kr
        else:
            k_left, k_right = kl, kr

        trans = ax.get_xaxis_transform()
        ax.text(x[i], -0.16, f"{left_name} k = {k_left}",
                ha="center", va="top", fontsize=8, transform=trans)
        ax.text(x[i], -0.24, f"{right_name} k = {k_right}",
                ha="center", va="top", fontsize=8, transform=trans)

    plt.subplots_adjust(bottom=0.32, right=0.78)

    fig.savefig(outbase.with_suffix(".png"))
    fig.savefig(outbase.with_suffix(".pdf"))
    plt.close(fig)


# =========================
# 図化: 方向比較 24 本棒グラフ
# =========================
def save_bar_24_compare(MET: pd.DataFrame,
                        metric: str,
                        title: str,
                        outbase: Path):
    """
    1 図の中に
      - OH→FP (top3, cumeig)
      - FP→OH (top3, cumeig)
    の 4 本を NbClust 指標ごとに並べた 24 本棒グラフ。
    x 軸下に OH k=?, FP k=? を表示（OH→FP, mode=top3 の k_left/k_right を代表値とする）
    """
    sub = MET.copy()
    for col in ["k_left", "k_right", metric]:
        if col not in sub.columns:
            sub[col] = np.nan

    x = np.arange(len(IDX_ORDER), dtype=float)
    width = 0.18

    vals_OH_top, vals_OH_cum = [], []
    vals_FP_top, vals_FP_cum = [], []
    k_oh_top, k_fp_top = [], []

    for ix in IDX_ORDER:
        r_OH_t = sub[(sub["direction"] == "OH→FP") & (sub["index"] == ix) & (sub["mode"] == "top3")]
        r_OH_c = sub[(sub["direction"] == "OH→FP") & (sub["index"] == ix) & (sub["mode"] == "cumeig")]
        r_FP_t = sub[(sub["direction"] == "FP→OH") & (sub["index"] == ix) & (sub["mode"] == "top3")]
        r_FP_c = sub[(sub["direction"] == "FP→OH") & (sub["index"] == ix) & (sub["mode"] == "cumeig")]

        def get_val(row):
            if row.empty:
                return np.nan
            v = row.iloc[0][metric]
            return float(v) if pd.notna(v) else np.nan

        vals_OH_top.append(get_val(r_OH_t))
        vals_OH_cum.append(get_val(r_OH_c))
        vals_FP_top.append(get_val(r_FP_t))
        vals_FP_cum.append(get_val(r_FP_c))

        if r_OH_t.empty:
            k_oh_top.append(np.nan)
            k_fp_top.append(np.nan)
        else:
            kl = r_OH_t.iloc[0]["k_left"]
            kr = r_OH_t.iloc[0]["k_right"]
            k_oh_top.append(kl)
            k_fp_top.append(kr)

    fig, ax = plt.subplots(figsize=(11.0, 5.0), dpi=300)

    ax.bar(x - 1.5 * width, vals_OH_top, width=width, color=COLOR_OH_TOP3, label="OH→FP (top3)")
    ax.bar(x - 0.5 * width, vals_OH_cum, width=width, color=COLOR_OH_CUMEIG, label="OH→FP (cumeig)")
    ax.bar(x + 0.5 * width, vals_FP_top, width=width, color=COLOR_FP_TOP3, label="FP→OH (top3)")
    ax.bar(x + 1.5 * width, vals_FP_cum, width=width, color=COLOR_FP_CUMEIG, label="FP→OH (cumeig)")

    ax.set_xticks(x)
    ax.set_xticklabels(IDX_ORDER, rotation=0, ha="center")
    ax.set_ylabel(metric)
    ax.set_title(title)

    finite_vals = [v for v in (vals_OH_top + vals_OH_cum + vals_FP_top + vals_FP_cum) if np.isfinite(v)]
    if finite_vals:
        vmax = max(finite_vals)
    else:
        vmax = 1.0

    if metric == "pair_mean_JS_divergence":
        ytop = float(min(max(vmax * 1.20, 0.2), 2.0))
        ax.set_ylim(0.0, ytop)
    elif metric == "mean_entropy":
        ytop = float(min(max(vmax * 1.20, 0.5), 6.0))
        ax.set_ylim(0.0, ytop)
    else:
        ytop = float(min(max(vmax * 1.10, 0.4), 1.05))
        ax.set_ylim(0.0, ytop)

    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5, axis="y")

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.0)

    # x 軸下に OH/FP の k を表示（代表値: OH→FP, top3）
    trans = ax.get_xaxis_transform()
    for i, ix in enumerate(IDX_ORDER):
        ohk = k_oh_top[i]
        fpk = k_fp_top[i]
        if pd.isna(ohk) or pd.isna(fpk):
            continue
        ax.text(x[i], -0.16, f"OH k = {ohk}", ha="center", va="top", fontsize=8, transform=trans)
        ax.text(x[i], -0.24, f"FP k = {fpk}", ha="center", va="top", fontsize=8, transform=trans)

    plt.subplots_adjust(bottom=0.32, right=0.78)

    fig.savefig(outbase.with_suffix(".png"))
    fig.savefig(outbase.with_suffix(".pdf"))
    plt.close(fig)


# =========================
# メイン
# =========================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", type=str, default=str(DEFAULT_BASE),
                    help="sub のベースディレクトリ（03_clustering_STEP3.2_signlessCorr がある場所）")
    ap.add_argument("--featroot", type=str, default=str(DEFAULT_FEATROOT),
                    help="preprocessed_features_{OH,FP}.csv のあるルート（for_MDS_STEP2）")
    ap.add_argument("--mds_method", type=str, default="linear", choices=["linear", "nonlinear"],
                    help="cluster_labels_matrix のどちらの MDS method を使うか（prefix）")
    args, _ = ap.parse_known_args()

    BASE = Path(args.base).resolve()
    FEAT_ROOT = Path(args.featroot).resolve()
    mds_method = args.mds_method

    OUT_DIR = BASE / "plots_sixmetrics_sub"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("[INFO] BASE (sub for STEP3.2 results):", BASE)
    print("[INFO] FEAT_ROOT (for_MDS_STEP2):      ", FEAT_ROOT)
    print("[INFO] Output dir:", OUT_DIR)
    print("[INFO] MDS method used for assignments:", mds_method)

    # 1) 割当（cluster_labels_matrix → 縦持ち化）
    print("[INFO] Searching cluster_labels_matrix (variables OH/FP) under BASE...")
    A_OH = read_assignments_from_sub(BASE, "OH", mds_method=mds_method)
    A_FP = read_assignments_from_sub(BASE, "FP", mds_method=mds_method)

    if A_OH.empty or A_FP.empty:
        print("❌ 割当情報が取得できませんでした。cluster_labels_matrix の有無を確認してください。")
        return

    # 2) 特徴量（列=変数 ➜ 転置して index=変数）→ 負値0クリップ → 行確率化
    X_OH = load_features_columns_as_variables(FEAT_ROOT / "preprocessed_features_OH.csv")
    X_FP = load_features_columns_as_variables(FEAT_ROOT / "preprocessed_features_FP.csv")
    print("[INFO] Features (variables x samples):", X_OH.shape, X_FP.shape)

    if X_OH.empty or X_FP.empty:
        print("❌ 特徴量CSVが読み込めませんでした。--featroot を確認してください。")
        return

    # 共通 id（変数名）のみ使用
    common_ids = sorted(
        set(A_OH["id"]) &
        set(A_FP["id"]) &
        set(X_OH.index.astype(str)) &
        set(X_FP.index.astype(str))
    )
    print(f"[INFO] common ids (variables) = {len(common_ids)}")
    if len(common_ids) == 0:
        print("❌ 共通の変数名が 0 件です。割当CSVの id と特徴量CSVの列名が一致しているか確認してください。")
        return

    A_OH = A_OH[A_OH["id"].isin(common_ids)].copy()
    A_FP = A_FP[A_FP["id"].isin(common_ids)].copy()
    X_OH = X_OH.loc[common_ids]
    X_FP = X_FP.loc[common_ids]
    P_OH = row_normalize_to_prob(X_OH)
    P_FP = row_normalize_to_prob(X_FP)

    # 3) 6 指標（両方向）
    rows = []
    for md in MODES:
        for ix in IDX_ORDER:
            a = six_metrics_direction(A_OH, A_FP, P_FP, md, ix)  # OH→FP
            b = six_metrics_direction(A_FP, A_OH, P_OH, md, ix)  # FP→OH
            if a:
                rows.append({"direction": "OH→FP", "mode": md, "index": ix, **a})
            if b:
                rows.append({"direction": "FP→OH", "mode": md, "index": ix, **b})

    if not rows:
        print("❌ 6指標が計算できませんでした（データ対応の不整合の可能性）。")
        return

    MET = pd.DataFrame(rows)

    # 4) 正方向にそろえた補助指標 & composite5
    with np.errstate(divide="ignore", invalid="ignore"):
        kr = MET["k_right"].clip(lower=2)
        maxH = np.log2(kr)
        MET["entropy_coh"] = 1.0 - (MET["mean_entropy"] / maxH)
    MET["entropy_coh"] = MET["entropy_coh"].clip(0, 1)

    MET["js_coh"] = 1.0 - MET["pair_mean_JS_divergence"]
    MET["js_coh"] = MET["js_coh"].clip(0, 1)

    MET["composite5"] = MET[
        [
            "mean_purity",
            "pair_major_same_rate",
            "pair_mean_cosine_similarity",
            "cluster_median_cohesive_ratio",
            "entropy_coh",
        ]
    ].mean(axis=1)

    # 5) 保存
    out_csv = OUT_DIR / "Table_sixmetrics_bidirectional_composite5_sub.csv"
    MET.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("✅ Saved table:", out_csv)

    # 6) 図化
    metrics = [
        "mean_purity",
        "pair_major_same_rate",
        "pair_mean_cosine_similarity",
        "cluster_median_cohesive_ratio",
        "mean_entropy",            # 参考（小さいほど整合的）
        "pair_mean_JS_divergence", # 参考（小さいほど整合的）
        "composite5",              # 主指標（大きいほど整合的）
    ]

    # (1) 方向別 12 本棒グラフ
    for direction in ["OH→FP", "FP→OH"]:
        sub = MET[MET["direction"] == direction].copy()
        for m in metrics:
            title = f"{direction}: {m} (top3 & cumeig by NbClust index)"
            outbase = OUT_DIR / f"Fig_{direction.replace('→','to')}_{m}_top3_cumeig_sub"
            save_bar_12(sub, m, title, direction, outbase)

    # (2) 方向比較 24 本棒グラフ
    for m in metrics:
        title = f"OH→FP vs FP→OH: {m} (top3 & cumeig by NbClust index)"
        outbase = OUT_DIR / f"Fig_compareDirections_{m}_top3_cumeig_sub"
        save_bar_24_compare(MET, m, title, outbase)

    print("✅ Saved figures (PNG/PDF) to:", OUT_DIR)


if __name__ == "__main__":
    main()

[INFO] BASE (sub for STEP3.2 results): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub
[INFO] FEAT_ROOT (for_MDS_STEP2):       /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2
[INFO] Output dir: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/plots_sixmetrics_sub
[INFO] MDS method used for assignments: linear
[INFO] Searching cluster_labels_matrix (variables OH/FP) under BASE...
[INFO] Using cluster_labels_matrix for OH: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500/variables/cluster_labels_matrix_variables_OH_20251130_153500.csv
[WARN] 'id' column not found. Use 'ID' as id.
[DEBUG]   skip col